# 02 - Inferential statistics

Hypotheses, assumption checks, effect sizes, CIs and BH correction (`src/ecom/stats`). The first cells re-run one test by hand to show the mechanics.

In [1]:
import json
import pandas as pd
from IPython.display import Image, Markdown, display
from ecom.config import get_config
from ecom.layout import outputs_dir, table_path
cfg = get_config()
OUT = outputs_dir(cfg)
FIG = cfg.paths.figures
pd.set_option("display.float_format", "{:,.4f}".format)
print(f"source={cfg.source} sample={cfg.sample_rate:.0%} outputs={OUT}")

source=synthetic sample=100% outputs=D:\smaiproj\data\outputs\synthetic\full


## Weekend vs weekday conversion, step by step

In [2]:
from ecom.stats.core import two_proportion_ztest
from ecom.stats.hypothesis_tests import connect
con = connect(cfg)
t = con.execute('SELECT is_weekend, COUNT(*) n, SUM(purchased::INT) x FROM sessions GROUP BY 1').df().set_index('is_weekend')
t

,n,x
is_weekend,,
True,661991,"57,611.0000"
False,1372703,"95,970.0000"


In [3]:
res = two_proportion_ztest(int(t.loc[False, 'x']), int(t.loc[False, 'n']),
                          int(t.loc[True, 'x']), int(t.loc[True, 'n']))
print(res.h0, '| z =', round(res.statistic, 2), '| p =', f'{res.p_value:.2g}')
print('diff CI:', round(res.ci_low, 4), round(res.ci_high, 4), '| relative lift:', f"{res.details['relative_lift']:+.1%}")

p_A = p_B | z = 43.29 | p = 0
diff CI: 0.0163 0.0179 | relative lift: +24.5%


## All tests (BH-adjusted)

In [4]:
pd.read_csv(OUT / "stats_tests.csv")

,test,method,statistic,p_value,p_bh,effect_size,effect,ci_low,ci_high,ci_of,n,significant_bh
0,Session conversion: weekend vs weekday,two-proportion z-test,43.2944,0.0000,0.0000,0.0637,Cohen's h,0.0163,0.0179,p_B - p_A,2034694,True
1,Session conversion x weekend (chi-square),Pearson chi-square test of independence,"1,874.4055",0.0000,0.0000,0.0304,phi,NaN,NaN,NaN,2034694,True
2,Order value across top 5 categories,Kruskal-Wallis H (ANOVA assumptions failed),"21,516.6208",0.0000,0.0000,0.3711,epsilon-squared,NaN,NaN,NaN,57982,True
3,Session duration: buyers vs non-buyers,Mann-Whitney U (two-sided),"1,905,824,030.0000",0.0000,0.0000,0.5247,rank-biserial r,157.0000,166.0000,median_B - median_A (bootstrap),100000,True
4,Product price vs view->purchase conversion (pr...,Spearman rank correlation,-0.2224,0.0000,0.0000,-0.2224,Spearman rho,-0.2334,-0.2113,"rho (Fisher z, Bonett-Wright SE)",28983,True


## Assumption checks and post-hoc tests

In [5]:
tests = json.loads((OUT / "stats_tests.json").read_text())["tests"]
for t in tests:
    print(t['name'], '->', t['test'])
    print('   assumptions:', t['assumptions'])

Session conversion: weekend vs weekday -> two-proportion z-test
   assumptions: {'independent_units': 'approx. - sessions of one user are correlated; effect is large relative to clustering', 'min_expected_cell_>=10': True, 'min_cell': 57611}
Session conversion x weekend (chi-square) -> Pearson chi-square test of independence
   assumptions: {'min_expected_>=5': True, 'dof': 1}
Order value across top 5 categories -> Kruskal-Wallis H (ANOVA assumptions failed)
   assumptions: {'normal_by_group': {'electronics': False, 'appliances': False, 'computers': False, 'apparel': False, 'construction': False}, 'levene_p': 0.0, 'equal_variances': False}
Session duration: buyers vs non-buyers -> Mann-Whitney U (two-sided)
   assumptions: {'normal_A': False, 'normal_B': False, 'independent_samples': True}
Product price vs view->purchase conversion (products with >= 30 views) -> Spearman rank correlation
   assumptions: {'monotonic_relationship_assumed': True, 'n_>=10': True}


In [6]:
aov = next(t for t in tests if t['name'].startswith('Order value'))
pd.DataFrame(aov['details']['posthoc_dunn_bh'])

,a,b,z,p,p_adj,significant
0,electronics,appliances,-26.8350,0.0000,0.0000,True
1,electronics,computers,-99.8285,0.0000,0.0000,True
2,electronics,apparel,63.6919,0.0000,0.0000,True
3,electronics,construction,38.4679,0.0000,0.0000,True
4,appliances,computers,-78.1280,0.0000,0.0000,True
5,appliances,apparel,81.4509,0.0000,0.0000,True
6,appliances,construction,52.5778,0.0000,0.0000,True
7,computers,apparel,131.1948,0.0000,0.0000,True
8,computers,construction,96.8487,0.0000,0.0000,True
9,apparel,construction,-10.2940,0.0000,0.0000,True


## Bootstrap CIs for the median order value

In [7]:
pd.read_csv(OUT / "bootstrap_median_aov.csv")

,segment,median,ci_low,ci_high,n
0,all orders,168.9600,168.4399,169.5000,117996
1,electronics,167.9800,167.5900,168.3301,20000
2,appliances,226.5200,224.8320,228.4000,19724
3,computers,524.3250,511.6738,534.6706,9390
4,apparel,54.4300,54.1000,54.7300,5659
5,construction,74.4500,69.8700,77.8245,3209
